In [ ]:
# In[1]:

import pandas as pd

# Load CSVs
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_df = pd.read_csv("error_logs.csv")

# Ensure timestamp columns are numeric integers and also create datetime (UTC) columns for internal use per rules
for df in (metric_df, trace_df, log_df, error_df):
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce').astype('Int64')
        df['timestamp_dt'] = pd.to_datetime(df['timestamp'].astype('Int64'), unit='s', utc=True)

# Ensure value columns are numeric where present
for df, col in ((metric_df, 'value'), (trace_df, 'value'), (log_df, 'value')):
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# 1) metric.csv summary
metric_grp = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .agg(count='count',
         p95_value=lambda s: s.quantile(0.95))
    .reset_index()
)
# compute min/max timestamps per group (use original timestamp integers)
ts_min_max = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['timestamp']
    .agg(min_timestamp='min', max_timestamp='max')
    .reset_index()
)
metric_summary = (
    metric_grp.merge(ts_min_max, on=['cmdb_id', 'kpi_name'])
    .sort_values('count', ascending=False)
    .loc[:, ['cmdb_id', 'kpi_name', 'count', 'min_timestamp', 'max_timestamp', 'p95_value']]
    .reset_index(drop=True)
    .head(50)
)

# 2) trace.csv summary
trace_grp = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], dropna=False)['value']
    .agg(count='count',
         p95_value=lambda s: s.quantile(0.95))
    .reset_index()
)
ts_min_max_trace = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], dropna=False)['timestamp']
    .agg(min_timestamp='min', max_timestamp='max')
    .reset_index()
)
trace_summary = (
    trace_grp.merge(ts_min_max_trace, on=['cmdb_id', 'trace_name'])
    .sort_values('count', ascending=False)
    .loc[:, ['cmdb_id', 'trace_name', 'count', 'min_timestamp', 'max_timestamp', 'p95_value']]
    .reset_index(drop=True)
    .head(50)
)

# 3) log.csv summary
log_grp = (
    log_df
    .groupby(['cmdb_id', 'log_name'], dropna=False)['value']
    .agg(count='count',
         p95_value=lambda s: s.quantile(0.95))
    .reset_index()
)
ts_min_max_log = (
    log_df
    .groupby(['cmdb_id', 'log_name'], dropna=False)['timestamp']
    .agg(min_timestamp='min', max_timestamp='max')
    .reset_index()
)
log_summary = (
    log_grp.merge(ts_min_max_log, on=['cmdb_id', 'log_name'])
    .sort_values('count', ascending=False)
    .loc[:, ['cmdb_id', 'log_name', 'count', 'min_timestamp', 'max_timestamp', 'p95_value']]
    .reset_index(drop=True)
    .head(50)
)

# 4) error_logs.csv overview
total_error_count = int(len(error_df))
unique_cmdb_full = pd.Series(error_df['cmdb_id'].dropna().unique())
unique_cmdb_list = unique_cmdb_full[:20].tolist()
min_error_ts = int(error_df['timestamp'].min()) if not error_df['timestamp'].isna().all() else pd.NA
max_error_ts = int(error_df['timestamp'].max()) if not error_df['timestamp'].isna().all() else pd.NA

# Up to top 5 distinct message samples: keep first occurrence of each distinct message, order by timestamp
if 'message' in error_df.columns:
    error_samples = (
        error_df
        .dropna(subset=['message'])
        .drop_duplicates(subset=['message'], keep='first')
        .sort_values('timestamp', ascending=True)
        .loc[:, ['timestamp', 'cmdb_id', 'message']]
        .head(5)
        .reset_index(drop=True)
    )
else:
    error_samples = pd.DataFrame(columns=['timestamp', 'cmdb_id', 'message'])

error_overview = pd.DataFrame([{
    'total_count': total_error_count,
    'unique_cmdb_count': int(unique_cmdb_full.size),
    'unique_cmdb_samples_limited': unique_cmdb_list,
    'min_timestamp': min_error_ts,
    'max_timestamp': max_error_ts
}])

# Display compact summaries (variables are the outputs; multiple variables separated by commas)
metric_summary, trace_summary, log_summary, error_overview, error_samples

```
Out[1]:
```
Summary of the telemetry (straightforward key findings):

- Data coverage
  - metric, trace and log series show regular samples (most groups have count = 25, i.e., a ~25-minute window).
  - Global timestamp range shown in the summaries: min = 1706045400, max = 1706046840.

- error_logs
  - total_count = 0 (no error log rows found), so no error-log samples to inspect.

- Metric highlights (top observations)
  - Very large disk I/O P95 values observed:
    - ts-admin-basic-info-service: diskio P95 ≈ 57,558.85
    - ts-basic-service: diskio P95 ≈ 59,669.53
  - Elevated CPU P95 for some services:
    - ts-basic-service: cpu P95 ≈ 6.67
    - ts-config-service: cpu P95 ≈ 1.45
  - Socket metric: ts-voucher-service socket P95 = 4.0
  - Many metric groups are present and stable across the window (counts ≈ 24–25).

- Trace highlights
  - High trace row_count P95 to order-service:
    - trace.to_ts-order-service.row_count P95 ≈ 180.2 (large traffic/trace-count)
    - trace.to_ts-order-service.duration_p95 ≈ 0.370 s
  - trace.to_ts-user-service.row_count P95 ≈ 12
  - Most duration P95s are small (<1s) but the order-service trace counts and p95 duration stand out.

- Log highlights
  - log.row_count P95 is nontrivial for some services (examples):
    - ts-consign-service log.row_count P95 ≈ 61.95
    - ts-admin-basic-info-service log.row_count P95 ≈ 32.60
  - log.error_count P95 is 0 for the top services (no elevated error-count in logs).

- Immediate interpretation / next steps
  - There are no error_logs to inspect, so focus on metrics/traces:
    - Investigate high disk I/O on ts-admin-basic-info-service and ts-basic-service.
    - Investigate high trace row_count and increased duration for ts-order-service (possible load/latency contributor).
    - Check services with elevated log.row_count (ts-consign-service, ts-admin-basic-info-service) for increased activity.
  - These services (ts-admin-basic-info-service, ts-basic-service, ts-order-service, ts-consign-service, ts-voucher-service) are top candidates for further root-cause investigation.

If you want, I can: extract the top N metric/trace series for a more focused drill-down, show raw time slices for any of the noted services, or convert those Unix timestamps to readable UTC datetimes. Which would you prefer next?

The original code execution output of IPython Kernel is also provided below for reference:

(                        cmdb_id    kpi_name  count  min_timestamp  max_timestamp     p95_value
0            ts-voucher-service      socket     25     1706045400     1706046840      4.000000
1   ts-admin-basic-info-service         cpu     25     1706045400     1706046840      0.527695
2   ts-admin-basic-info-service      diskio     25     1706045400     1706046840  57558.847395
3   ts-admin-basic-info-service       error     25     1706045400     1706046840      0.000000
4   ts-admin-basic-info-service  latency-50     25     1706045400     1706046840      0.016968
..                          ...         ...    ...            ...            ...           ...
45            ts-config-service         cpu     25     1706045400     1706046840      1.451366
46             ts-basic-service         cpu     25     1706045400     1706046840      6.666644
47             ts-basic-service      diskio     25     1706045400     1706046840  59669.529276
48             ts-basic-service  latency-50     25     1706045400     1706046840      0.008231
49             ts-basic-service  latency-90     25     1706045400     1706046840      0.033218

[50 rows x 6 columns],             cmdb_id                                         trace_name  count  min_timestamp  max_timestamp   p95_value
0   ts-user-service                 trace.to_ts-user-service.row_count     25     1706045400     1706046840   12.000000
1              root  trace.to_ts-admin-basic-info-service.duration_...     25     1706045400     1706046840    0.010454
2              root  trace.to_ts-admin-basic-info-service.duration_p95     25     1706045400     1706046840    0.015401
3              root    trace.to_ts-admin-basic-info-service.error_rate     25     1706045400     1706046840    0.000000
4              root     trace.to_ts-admin-basic-info-service.row_count     25     1706045400     1706046840   20.800000
..              ...                                                ...    ...            ...            ...         ...
45             root             trace.to_ts-order-service.duration_p95     25     1706045400     1706046840    0.370260
46             root               trace.to_ts-order-service.error_rate     25     1706045400     1706046840    0.000000
47             root                trace.to_ts-order-service.row_count     25     1706045400     1706046840  180.200000
48             root             trace.to_ts-food-service.duration_mean     25     1706045400     1706046840    0.027178
49             root              trace.to_ts-food-service.duration_p95     25     1706045400     1706046840    0.058531

[50 rows x 6 columns],                         cmdb_id         log_name  count  min_timestamp  max_timestamp  p95_value
0   ts-admin-basic-info-service  log.error_count     25     1706045400     1706046840       0.00
1   ts-admin-basic-info-service    log.row_count     25     1706045400     1706046840      32.60
2       ts-admin-travel-service  log.error_count     25     1706045400     1706046840       0.00
3       ts-admin-travel-service    log.row_count     25     1706045400     1706046840      32.40
4          ts-assurance-service    log.row_count     25     1706045400     1706046840      14.80
..                          ...              ...    ...            ...            ...        ...
45            ts-contacts-mongo    log.row_count     24     1706045400     1706046780       1.85
46       ts-consign-price-mongo    log.row_count     24     1706045460     1706046840       1.85
47       ts-consign-price-mongo  log.error_count     24     1706045460     1706046840       0.00
48           ts-consign-service  log.error_count     24     1706045400     1706046840       0.00
49           ts-consign-service    log.row_count     24     1706045400     1706046840      61.95

[50 rows x 6 columns],    total_count  unique_cmdb_count unique_cmdb_samples_limited min_timestamp max_timestamp
0            0                  0                          []          <NA>          <NA>, Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd

# Reuse metric_df from prior state
# 1) Global P95 and total global count per (cmdb_id, kpi_name)
metric_global = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .agg(global_p95=lambda s: float(s.quantile(0.95)), total_count_global='count')
    .reset_index()
)

# 2) Incident window timestamps (UTC) inclusive
start_ts = int(pd.Timestamp("2024-01-23T21:27:22Z", tz='UTC').timestamp())
end_ts   = int(pd.Timestamp("2024-01-23T21:57:22Z", tz='UTC').timestamp())

metric_window = metric_df[(metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] <= end_ts)].copy()

if not metric_window.empty:
    # 3) Merge global thresholds, mark anomalies
    metric_window = metric_window.merge(metric_global, on=['cmdb_id', 'kpi_name'], how='left')
    metric_window['is_anomaly'] = metric_window['value'] >= metric_window['global_p95']

    # Aggregates for window
    agg_window = (
        metric_window
        .groupby(['cmdb_id', 'kpi_name'], dropna=False)
        .agg(
            count_in_window = ('value', 'count'),
            count_anomalies_in_window = ('is_anomaly', 'sum'),
            max_value_in_window = ('value', 'max')
        )
        .reset_index()
    )

    # First anomaly timestamp per group (from anomaly rows only)
    first_anom = (
        metric_window[metric_window['is_anomaly']]
        .groupby(['cmdb_id', 'kpi_name'], dropna=False)['timestamp']
        .min()
        .reset_index()
        .rename(columns={'timestamp': 'first_anomaly_timestamp'})
    )

    # Combine window aggregates with first anomaly timestamps
    window_summary = agg_window.merge(first_anom, on=['cmdb_id', 'kpi_name'], how='left')

    # Merge with global stats
    anomalies_summary = (
        window_summary
        .merge(metric_global, on=['cmdb_id', 'kpi_name'], how='left')
        .loc[:, ['cmdb_id', 'kpi_name', 'global_p95', 'total_count_global',
                 'count_in_window', 'count_anomalies_in_window',
                 'first_anomaly_timestamp', 'max_value_in_window']]
    )

    # Ensure types and missing handling
    anomalies_summary['first_anomaly_timestamp'] = anomalies_summary['first_anomaly_timestamp'].astype('Int64')
    anomalies_summary['total_count_global'] = anomalies_summary['total_count_global'].astype(int)

    # 4) Filter groups with anomalies and limit top 20 by count_anomalies_in_window desc
    anomalies_summary = (
        anomalies_summary[anomalies_summary['count_anomalies_in_window'] > 0]
        .sort_values('count_anomalies_in_window', ascending=False)
        .reset_index(drop=True)
        .head(20)
    )
else:
    anomalies_summary = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','global_p95','total_count_global',
        'count_in_window','count_anomalies_in_window',
        'first_anomaly_timestamp','max_value_in_window'
    ])

anomalies_summary

```
Out[2]:
```
Summary of anomaly detection (metric.csv) for window 2024-01-23T21:27:22Z — 2024-01-23T21:57:22Z:

- Overall result: 20 (cmdb_id, kpi_name) groups were flagged. For each group:
  - total_count_global = 25, count_in_window = 25, count_anomalies_in_window = 25 — meaning every sampled point in the incident window exceeded the group's global P95.
  - first_anomaly_timestamp = 1706045400 (the earliest unix timestamp in the window where the threshold was exceeded) for all groups.
- Top affected groups (examples from the result):
  - ts-ui-dashboard — error (global_p95 = 0.0), max_value_in_window = 0.0
  - ts-travel-mongo — socket (global_p95 = 6.0), max = 6.0
  - ts-avatar-service — diskio (global_p95 = 0.0) and socket (global_p95 = 5.0)
  - ts-ticket-office-mongo / service — socket (2.0 / 1.0) and diskio (0.0)
  - ts-config-mongo, ts-order-mongo, ts-payment-mongo, ts-preserve-mongo, ts-notification-mongo, ts-order-other-mongo, ts-assurance-mongo — mostly socket metrics flagged
  - ts-admin-basic-info-service / ts-admin-travel-service — error and latency-50 metrics flagged
  - ts-voucher-service — socket flagged

Interpretation and recommended next steps:
- The anomaly pattern is broad and consistent: many "socket" metrics across multiple mongo and service components exceed their global P95 throughout the entire incident window — this suggests a systemic, sustained shift (e.g., increased socket usage/congestion or a changed workload) rather than isolated spikes.
- Some diskio, error, and latency-50 metrics are also persistently above their historical P95 for the window.
- Actions to take now:
  1. Inspect services with repeated socket anomalies (mongo-related components and gateway/service sockets) for network/socket saturation, connection errors, or connection pool exhaustion.
  2. Check ts-admin-basic-info-service and ts-admin-travel-service for error and latency issues.
  3. Review recent deployment/config changes and traffic pattern changes that could produce a sustained shift.
  4. Correlate with traces/logs (if available) and infrastructure metrics (CPU, network, disk I/O) for those services.

If you want, I can: show the exact rows in the incident window for any specific (cmdb_id, kpi_name), convert 1706045400 to a human-readable UTC time, or compute per-group severity scores. Which would you like next?

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name  global_p95  total_count_global  count_in_window  count_anomalies_in_window  first_anomaly_timestamp  max_value_in_window
0               ts-ui-dashboard       error       0.000                  25               25                         25               1706045400                0.000
1               ts-travel-mongo      socket       6.000                  25               25                         25               1706045400                6.000
2             ts-avatar-service      diskio       0.000                  25               25                         25               1706045400                0.000
3               ts-config-mongo      socket       6.000                  25               25                         25               1706045400                6.000
4             ts-avatar-service      socket       5.000                  25               25                         25               1706045400                5.000
5        ts-ticket-office-mongo      socket       2.000                  25               25                         25               1706045400                2.000
6      ts-ticket-office-service      socket       1.000                  25               25                         25               1706045400                1.000
7             ts-security-mongo      socket       4.000                  25               25                         25               1706045400                4.000
8       ts-preserve-other-mongo      socket       2.000                  25               25                         25               1706045400                2.000
9      ts-ticket-office-service      diskio       0.000                  25               25                         25               1706045400                0.000
10               ts-order-mongo      socket       6.000                  25               25                         25               1706045400                6.000
11        ts-notification-mongo      socket       2.000                  25               25                         25               1706045400                2.000
12            ts-preserve-mongo      socket       2.000                  25               25                         25               1706045400                2.000
13             ts-payment-mongo      socket       4.000                  25               25                         25               1706045400                4.000
14         ts-order-other-mongo      socket       7.000                  25               25                         25               1706045400                7.000
15           ts-assurance-mongo      socket       4.000                  25               25                         25               1706045400                4.000
16  ts-admin-basic-info-service       error       0.000                  25               25                         25               1706045400                0.000
17      ts-admin-travel-service  latency-50       0.175                  25               25                         25               1706045400                0.175
18           ts-voucher-service      socket       4.000                  25               25                         25               1706045400                4.000
19      ts-admin-travel-service       error       0.000                  25               25                         25               1706045400                0.000```
```

In [ ]:
# In[3]:

import pandas as pd

# Reuse trace_df loaded earlier. Ensure numeric types.
trace_df['timestamp'] = pd.to_numeric(trace_df['timestamp'], errors='coerce').astype('Int64')
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# 1) Global P95 and full-series count per trace_name (computed on full series)
trace_global = (
    trace_df
    .groupby('trace_name', dropna=False)['value']
    .agg(global_p95=lambda s: float(s.quantile(0.95)), total_count_global='count')
    .reset_index()
)

# 2) Incident window (inclusive)
start_ts = 1706045242
end_ts   = 1706047442
trace_window = trace_df[(trace_df['timestamp'] >= start_ts) & (trace_df['timestamp'] <= end_ts)].copy()

# 3) If window not empty, compute per-trace window aggregates and anomalies
if not trace_window.empty:
    # Merge global thresholds
    trace_window = trace_window.merge(trace_global, on='trace_name', how='left')
    trace_window['is_anomaly'] = trace_window['value'] >= trace_window['global_p95']
    
    # Aggregates in window
    agg_window = (
        trace_window
        .groupby('trace_name', dropna=False)
        .agg(
            count_in_window = ('value', 'count'),
            count_anomalies_in_window = ('is_anomaly', 'sum'),
            max_value_in_window = ('value', 'max')
        )
        .reset_index()
    )
    
    # First anomaly timestamp per trace_name
    first_anom = (
        trace_window[trace_window['is_anomaly']]
        .groupby('trace_name', dropna=False)['timestamp']
        .min()
        .reset_index()
        .rename(columns={'timestamp': 'first_anomaly_timestamp'})
    )
    
    # Combine
    anomalies_summary = (
        agg_window.merge(first_anom, on='trace_name', how='left')
        .merge(trace_global, on='trace_name', how='left')
        .loc[:, ['trace_name', 'global_p95', 'total_count_global',
                 'count_in_window', 'count_anomalies_in_window',
                 'first_anomaly_timestamp', 'max_value_in_window']]
    )
    
    # Types and ordering; keep Unix seconds and Int64 for nullable ints
    anomalies_summary['first_anomaly_timestamp'] = anomalies_summary['first_anomaly_timestamp'].astype('Int64')
    anomalies_summary['total_count_global'] = anomalies_summary['total_count_global'].astype(int)
    
    # 4) Filter groups with anomalies and limit top 20 sorted by count_anomalies_in_window desc
    anomalies_summary = (
        anomalies_summary[anomalies_summary['count_anomalies_in_window'] > 0]
        .sort_values('count_anomalies_in_window', ascending=False)
        .reset_index(drop=True)
        .head(20)
    )
else:
    anomalies_summary = pd.DataFrame(columns=[
        'trace_name','global_p95','total_count_global',
        'count_in_window','count_anomalies_in_window',
        'first_anomaly_timestamp','max_value_in_window'
    ])

anomalies_summary

```
Out[3]:
```
Summary of trace-level anomaly detection for window 1706045242–1706047442 (2024-01-23T21:27:22Z–21:57:22Z):

Key findings
- Top flagged traces (20 shown) are all "error_rate" traces for many services (examples: trace.from_root.error_rate, trace.to_ts-order-service.error_rate, trace.to_ts-travel-service.error_rate, trace.to_ts-seat-service.error_rate, trace.to_ts-train-service.error_rate, etc.).
- For every reported trace:
  - global_p95 = 0.0
  - every sampled row in the incident window was counted as an anomaly (count_in_window == count_anomalies_in_window)
  - max_value_in_window = 0.0 for all top traces
  - first_anomaly_timestamp = 1706045400 for these groups
  - total_count_global varies by trace (examples: 664, 230, 191, 183, …)

Interpretation
- These are false-positive anomalies caused by the detection rule value >= global_p95 when global_p95 equals 0.0. Because trace error_rate values are 0.0 in the window and global_p95 is 0.0, the ">= global_p95" condition marks every zero as an anomaly.
- There is no evidence in the top results of non-zero error rates in the window (max_value_in_window = 0.0), so no real error-rate spikes are observed for these traces.

Recommended next steps
1. Fix anomaly criterion to avoid zero-threshold false positives (e.g., require value > global_p95 AND value > 0, or use a stricter rule such as value > global_p95 + epsilon, or use a different statistic like mean+std or IQR).
2. Re-run detection after adjusting thresholding logic.
3. If interested in other trace signals, run similar detection on duration or row_count traces (they may reveal latency/traffic anomalies).
4. Correlate with logs/metrics for traces that do show non-zero max values (none among the top 20 here) to confirm real issues.

If you want, I can: (a) re-run using a stricter anomaly rule (value > global_p95 and value > 0), (b) list traces whose max_value_in_window > 0, or (c) convert first_anomaly_timestamp to human-readable UTC times. Which do you prefer?

The original code execution output of IPython Kernel is also provided below for reference:

trace_name  global_p95  total_count_global  count_in_window  count_anomalies_in_window  first_anomaly_timestamp  max_value_in_window
0                        trace.from_root.error_rate         0.0                 664              664                        664               1706045400                  0.0
1         trace.from_ts-preserve-service.error_rate         0.0                 230              230                        230               1706045400                  0.0
2            trace.to_ts-station-service.error_rate         0.0                 191              191                        191               1706045400                  0.0
3   trace.from_ts-preserve-other-service.error_rate         0.0                 183              183                        183               1706045400                  0.0
4              trace.to_ts-order-service.error_rate         0.0                 170              170                        170               1706045400                  0.0
5        trace.to_ts-order-other-service.error_rate         0.0                 162              162                        162               1706045400                  0.0
6          trace.from_ts-travel2-service.error_rate         0.0                 150              150                        150               1706045400                  0.0
7           trace.from_ts-travel-service.error_rate         0.0                 150              150                        150               1706045400                  0.0
8             trace.from_ts-seat-service.error_rate         0.0                 150              150                        150               1706045400                  0.0
9             trace.to_ts-travel-service.error_rate         0.0                 146              146                        146               1706045400                  0.0
10              trace.to_ts-seat-service.error_rate         0.0                 140              140                        140               1706045400                  0.0
11        trace.to_ts-ticketinfo-service.error_rate         0.0                 139              139                        139               1706045400                  0.0
12           trace.from_ts-basic-service.error_rate         0.0                 125              125                        125               1706045400                  0.0
13             trace.to_ts-train-service.error_rate         0.0                 125              125                        125               1706045400                  0.0
14             trace.to_ts-route-service.error_rate         0.0                 125              125                        125               1706045400                  0.0
15           trace.to_ts-travel2-service.error_rate         0.0                 117              117                        117               1706045400                  0.0
16            trace.to_ts-config-service.error_rate         0.0                 100              100                        100               1706045400                  0.0
17            trace.from_ts-food-service.error_rate         0.0                 100              100                        100               1706045400                  0.0
18             trace.to_ts-price-service.error_rate         0.0                  99               99                         99               1706045400                  0.0
19          trace.to_ts-security-service.error_rate         0.0                  87               87                         87               1706045400                  0.0```
```

In [ ]:
# In[4]:

import pandas as pd

# Reuse metric_df from previous state; ensure types
metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce').astype('Int64')
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# Target cmdb_ids and incident window (inclusive)
target_cmdb = ['ts-auth-service', 'ts-order-service', 'ts-route-service', 'ts-train-service', 'ts-travel-service']
start_ts = 1706045242
end_ts = 1706047442

# 1) Compute global P95 and full-series count per (cmdb_id, kpi_name) for only the target cmdb_ids (using full available series)
metric_global = (
    metric_df[metric_df['cmdb_id'].isin(target_cmdb)]
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .agg(global_p95=lambda s: float(s.quantile(0.95)), total_count_global='count')
    .reset_index()
)

# 2) Filter rows to the incident window and target cmdb_ids
metric_window = metric_df[
    (metric_df['cmdb_id'].isin(target_cmdb)) &
    (metric_df['timestamp'] >= start_ts) &
    (metric_df['timestamp'] <= end_ts)
].copy()

# 3) Merge global thresholds into window rows and mark anomalies using strict '>'
if not metric_window.empty:
    metric_window = metric_window.merge(metric_global, on=['cmdb_id', 'kpi_name'], how='left')
    metric_window['is_anomaly'] = metric_window['value'] > metric_window['global_p95']

    # Aggregates per (cmdb_id, kpi_name) in the window
    agg_window = (
        metric_window
        .groupby(['cmdb_id', 'kpi_name'], dropna=False)
        .agg(
            count_in_window = ('value', 'count'),
            count_anomalies_in_window = ('is_anomaly', 'sum'),
            max_value_in_window = ('value', 'max')
        )
        .reset_index()
    )

    # First anomaly timestamp per group (strict >)
    first_anom = (
        metric_window[metric_window['is_anomaly']]
        .groupby(['cmdb_id', 'kpi_name'], dropna=False)['timestamp']
        .min()
        .reset_index()
        .rename(columns={'timestamp': 'first_anomaly_timestamp'})
    )

    # Combine window aggregates with first anomaly timestamps and global stats
    anomalies_summary = (
        agg_window.merge(first_anom, on=['cmdb_id', 'kpi_name'], how='left')
        .merge(metric_global, on=['cmdb_id', 'kpi_name'], how='left')
        .loc[:, ['cmdb_id', 'kpi_name', 'global_p95', 'total_count_global',
                 'count_in_window', 'count_anomalies_in_window',
                 'first_anomaly_timestamp', 'max_value_in_window']]
    )

    # Ensure types and fill/int handling
    anomalies_summary['first_anomaly_timestamp'] = anomalies_summary['first_anomaly_timestamp'].astype('Int64')
    anomalies_summary['total_count_global'] = anomalies_summary['total_count_global'].astype(int)

    # 4) Filter groups with at least one anomaly (strict >) and limit to top 20 by anomaly count desc
    anomalies_summary = (
        anomalies_summary[anomalies_summary['count_anomalies_in_window'] > 0]
        .sort_values('count_anomalies_in_window', ascending=False)
        .reset_index(drop=True)
        .head(20)
    )
else:
    anomalies_summary = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','global_p95','total_count_global',
        'count_in_window','count_anomalies_in_window',
        'first_anomaly_timestamp','max_value_in_window'
    ])

anomalies_summary

```
Out[4]:
```
Summary of metric anomalies for the five targeted services (window 1706045242–1706047442 UTC):

Key result
- Anomalies were detected only for ts-auth-service, ts-order-service, and ts-route-service. No groups from ts-train-service or ts-travel-service exceeded their global P95 (strict >) in the window.
- The output contains 20 (cmdb_id, kpi_name) groups. Every reported group has:
  - total_count_global = 25 (full series count),
  - count_in_window = 25,
  - count_anomalies_in_window = 2 (exactly two samples in the incident window strictly exceeded the global P95).
  - first_anomaly_timestamp varies by group (examples below),
  - max_value_in_window shows the peak observed in-window.

Notable groups / examples
- ts-auth-service:
  - cpu:    global_p95 ≈ 17.3826, count_anomalies_in_window = 2, first_anomaly_timestamp = 1706045700, max_value_in_window ≈ 18.16302
  - diskio: global_p95 ≈ 61,373.11, first_anomaly_timestamp = 1706045400, max ≈ 182,673.3
  - latency-50 / latency-90: small absolute latencies but exceeded their P95 twice (first anomaly timestamps 1706046240)
  - mem, socket, workload similarly had 2 anomaly points each.
- ts-order-service:
  - cpu:    global_p95 ≈ 2.55166, first_anomaly_timestamp = 1706045400, max ≈ 3.47119
  - diskio: global_p95 ≈ 58,642.42, first_anomaly_timestamp = 1706046000, max ≈ 60,259.80
  - latency-50/90, mem, socket, workload — each had 2 anomalous points.
- ts-route-service:
  - cpu, diskio, latency-50/90, mem, socket — each had 2 anomalous points (first anomalies at timestamps like 1706045400, 1706045640, 1706046660, 1706046780 depending on KPI).

Interpretation
- The anomalies are sparse but real: each flagged KPI exceeded its historical P95 at two sample times during the ~30-minute window. This looks like short-lived spikes (2 samples out of 25) rather than a sustained shift across the window.
- Disk I/O and memory peaks (notably for ts-auth-service and ts-order-service) stand out by absolute magnitude and may be the higher-risk items to investigate.
- CPU and socket increases are smaller in absolute terms but are present across multiple services.

Recommended next steps
1. Inspect metric datapoints at the reported first_anomaly_timestamp values (e.g., 1706045400, 1706045700, 1706046240, 1706046000, 1706046300, 1706046660, 1706046780) for the listed KPIs to see exact values/context.
2. Correlate those timestamps with logs and traces for the affected services (auth, order, route) to find errors, slow calls, or root causes.
3. Prioritize investigation of diskio and mem spikes for ts-auth-service and ts-order-service (these have large absolute maxima).
4. If you want, I can: (a) show the exact rows for any specific (cmdb_id, kpi_name) around the first_anomaly_timestamp, (b) convert those Unix timestamps to readable UTC times, or (c) compute per-group anomaly severity scores. Which would you like next?

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name    global_p95  total_count_global  count_in_window  count_anomalies_in_window  first_anomaly_timestamp  max_value_in_window
0    ts-auth-service         cpu  1.738260e+01                  25               25                          2               1706045700         1.816302e+01
1    ts-auth-service      diskio  6.137311e+04                  25               25                          2               1706045400         1.826733e+05
2    ts-auth-service  latency-50  3.687731e-01                  25               25                          2               1706046240         4.231788e-01
3    ts-auth-service  latency-90  8.483347e-01                  25               25                          2               1706046240         9.153810e-01
4    ts-auth-service         mem  2.528478e+08                  25               25                          2               1706045700         2.536679e+08
5    ts-auth-service      socket  1.787333e+01                  25               25                          2               1706045580         1.800000e+01
6    ts-auth-service    workload  2.959607e+00                  25               25                          2               1706045460         2.966867e+00
7   ts-order-service         cpu  2.551659e+00                  25               25                          2               1706045400         3.471192e+00
8   ts-order-service      diskio  5.864242e+04                  25               25                          2               1706046000         6.025980e+04
9   ts-order-service  latency-50  8.350771e-03                  25               25                          2               1706046360         1.034087e-02
10  ts-order-service  latency-90  4.022816e-02                  25               25                          2               1706045400         5.689263e-02
11  ts-order-service         mem  2.580815e+08                  25               25                          2               1706046780         2.581456e+08
12  ts-order-service      socket  1.258667e+01                  25               25                          2               1706046300         1.353333e+01
13  ts-order-service    workload  8.216539e+00                  25               25                          2               1706046600         8.441100e+00
14  ts-route-service         cpu  4.565186e+00                  25               25                          2               1706046660         4.681233e+00
15  ts-route-service      diskio  5.818998e+04                  25               25                          2               1706045640         5.944401e+04
16  ts-route-service  latency-50  3.281679e-03                  25               25                          2               1706046780         3.295713e-03
17  ts-route-service  latency-90  6.015199e-03                  25               25                          2               1706046780         6.219287e-03
18  ts-route-service         mem  2.558758e+08                  25               25                          2               1706046780         2.559059e+08
19  ts-route-service      socket  1.572667e+01                  25               25                          2               1706045400         1.600000e+01```
```

In [ ]:
# In[5]:

import pandas as pd

# Reuse trace_df from previous state; ensure types
trace_df['timestamp'] = pd.to_numeric(trace_df['timestamp'], errors='coerce').astype('Int64')
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# Candidate substrings to match in trace_name
candidates = ['order-service', 'auth-service', 'route-service']

# Find trace_names that contain any of the candidate substrings (search full series)
mask = trace_df['trace_name'].astype(str).str.contains('|'.join(candidates), regex=True)
matching_trace_names = trace_df.loc[mask, 'trace_name'].unique()

# 1) Compute global P95 and full-series count for each matching trace_name (use full series)
trace_global_selected = (
    trace_df[trace_df['trace_name'].isin(matching_trace_names)]
    .groupby('trace_name', dropna=False)['value']
    .agg(global_p95=lambda s: float(s.quantile(0.95)), total_count_global='count')
    .reset_index()
)

# 2) Incident window (inclusive)
start_ts = 1706045242
end_ts   = 1706047442

trace_window = trace_df[
    (trace_df['trace_name'].isin(matching_trace_names)) &
    (trace_df['timestamp'] >= start_ts) &
    (trace_df['timestamp'] <= end_ts)
].copy()

# 3) Compute window aggregates and anomalies using strict '>' against global_p95
if not trace_window.empty:
    trace_window = trace_window.merge(trace_global_selected, on='trace_name', how='left')
    trace_window['is_anomaly'] = trace_window['value'] > trace_window['global_p95']

    agg_window = (
        trace_window
        .groupby('trace_name', dropna=False)
        .agg(
            count_in_window = ('value', 'count'),
            count_anomalies_in_window = ('is_anomaly', 'sum'),
            max_value_in_window = ('value', 'max')
        )
        .reset_index()
    )

    first_anom = (
        trace_window[trace_window['is_anomaly']]
        .groupby('trace_name', dropna=False)['timestamp']
        .min()
        .reset_index()
        .rename(columns={'timestamp': 'first_anomaly_timestamp'})
    )

    anomalies_summary = (
        agg_window.merge(first_anom, on='trace_name', how='left')
        .merge(trace_global_selected, on='trace_name', how='left')
        .loc[:, ['trace_name', 'global_p95', 'total_count_global',
                 'count_in_window', 'count_anomalies_in_window',
                 'first_anomaly_timestamp', 'max_value_in_window']]
    )

    anomalies_summary['first_anomaly_timestamp'] = anomalies_summary['first_anomaly_timestamp'].astype('Int64')
    anomalies_summary['total_count_global'] = anomalies_summary['total_count_global'].astype(int)

    anomalies_summary = (
        anomalies_summary[anomalies_summary['count_anomalies_in_window'] > 0]
        .sort_values('count_anomalies_in_window', ascending=False)
        .reset_index(drop=True)
        .head(20)
    )
else:
    anomalies_summary = pd.DataFrame(columns=[
        'trace_name','global_p95','total_count_global',
        'count_in_window','count_anomalies_in_window',
        'first_anomaly_timestamp','max_value_in_window'
    ])

anomalies_summary

```
Out[5]:
```
Summary of trace-level anomalies for traces referencing order-service, auth-service, or route-service (window 1706045242–1706047442 UTC):

- Overall:
  - 18 trace_name groups were flagged (returned top ≤20). All groups exceeded their full-series P95 at least once in the incident window using strict "value > global_p95".

- Most affected traces (examples / highlights):
  - order-service
    - trace.to_ts-order-service.row_count — global_p95 = 231.75, total_count_global = 170, count_in_window = 170, count_anomalies_in_window = 9, first_anomaly_timestamp = 1706045520, max_value_in_window = 347.0
    - trace.to_ts-order-service.duration_mean — global_p95 = 0.025111, count_anomalies_in_window = 9, first_anomaly_timestamp = 1706045400, max_value_in_window = 0.032163
    - trace.to_ts-order-service.duration_p95 — global_p95 = 0.281176, count_anomalies_in_window = 9, first_anomaly_timestamp = 1706045400, max_value_in_window = 0.383226
    - trace.from_ts-order-service.row_count / duration_* also show anomalies (3 anomalies each, first_anomaly_timestamp ~1706045520 / 1706046600).
    - Interpretation: order-service experienced repeated traffic spikes (row_count) and corresponding latency spikes (duration_mean / p95).
  - route-service
    - trace.to_ts-route-service.duration_mean — global_p95 = 0.002981, count_anomalies_in_window = 7, first_anomaly_timestamp = 1706045820, max = 0.003413
    - trace.to_ts-route-service.duration_p95 — global_p95 = 0.004213, count_anomalies_in_window = 7, first_anomaly_timestamp = 1706045700, max = 0.009284
    - trace.to_ts-route-service.row_count — global_p95 = 956.0, count_anomalies_in_window = 6, first_anomaly_timestamp = 1706045460, max = 1470.0
    - Interpretation: route-service had several latency and traffic spikes (notably very high row_count peaks).
  - auth-service
    - trace.to_ts-auth-service.row_count — global_p95 = 210.8, count_anomalies_in_window = 3, first_anomaly_timestamp = 1706045460, max = 245.0
    - trace.to_ts-auth-service.duration_mean/p95 and trace.from_ts-auth-service.* show smaller but present duration anomalies (2–3 events; first_anomaly_timestamp values ~1706045400–1706046540).
    - Interpretation: auth-service shows moderate spike events in both load and latency.

- Practical interpretation:
  - order-service and route-service show the strongest signals: repeated row_count (traffic) spikes and associated latency increases — these are likely contributors to user-facing latency or capacity stress during the incident window.
  - auth-service shows smaller, fewer spikes but still notable latency/load deviations.
  - First anomaly times (Unix seconds) give actionable anchors for log/trace correlation (examples: 1706045400, 1706045460, 1706045520, 1706045700, 1706045820, 1706046060, 1706046300, 1706046420, 1706046540, 1706046600, 1706046780).

- Recommended next steps:
  1. Inspect full traces and logs for order-service and route-service at the listed first_anomaly_timestamp values (start with 1706045400 and 1706045520).
  2. Correlate the high row_count spikes with backend resource metrics (CPU, network, DB load) and any deployment/traffic changes.
  3. For auth-service, check connection/latency patterns around the reported first_anomaly_timestamp values.

If you want, I can: (a) show the exact trace rows around a specific first_anomaly_timestamp for a selected trace_name, or (b) convert any of the first_anomaly_timestamp values to readable UTC datetimes. Which would you like?

The original code execution output of IPython Kernel is also provided below for reference:

trace_name   global_p95  total_count_global  count_in_window  count_anomalies_in_window  first_anomaly_timestamp  max_value_in_window
0         trace.to_ts-order-service.row_count   231.750000                 170              170                          9               1706045520           347.000000
1     trace.to_ts-order-service.duration_mean     0.025111                 170              170                          9               1706045400             0.032163
2      trace.to_ts-order-service.duration_p95     0.281176                 170              170                          9               1706045400             0.383226
3     trace.to_ts-route-service.duration_mean     0.002981                 125              125                          7               1706045820             0.003413
4      trace.to_ts-route-service.duration_p95     0.004213                 125              125                          7               1706045700             0.009284
5         trace.to_ts-route-service.row_count   956.000000                 125              125                          6               1706045460          1470.000000
6      trace.to_ts-auth-service.duration_mean     0.198221                  50               50                          3               1706045400             0.237175
7       trace.from_ts-order-service.row_count   271.600000                  49               49                          3               1706045520           347.000000
8   trace.from_ts-order-service.duration_mean     0.389955                  49               49                          3               1706046600             0.396225
9    trace.from_ts-order-service.duration_p95     0.397782                  49               49                          3               1706046300             0.418359
10      trace.to_ts-auth-service.duration_p95     0.778147                  50               50                          3               1706046060             0.832137
11         trace.to_ts-auth-service.row_count   210.800000                  50               50                          3               1706045460           245.000000
12    trace.from_ts-auth-service.duration_p95     0.208318                  25               25                          2               1706046540             0.210287
13   trace.from_ts-auth-service.duration_mean     0.036147                  25               25                          2               1706046660             0.037211
14      trace.from_ts-route-service.row_count  1353.000000                  25               25                          2               1706045520          1470.000000
15   trace.from_ts-route-service.duration_p95     0.001363                  25               25                          2               1706046420             0.001390
16  trace.from_ts-route-service.duration_mean     0.000928                  25               25                          2               1706046780             0.000937
17       trace.from_ts-auth-service.row_count   239.000000                  25               25                          2               1706045520           245.000000```
```